# Fast activation recording with `tl.fastlog`

`tl.fastlog` is the sparse activation-recording path for cases where you know what you want to keep before the forward pass runs. `tl.log_forward_pass()` still builds a full `ModelLog` with graph metadata, labels, validation surfaces, and visualization. `save_new_activations()` is the fast second-pass path for an already traced `ModelLog` when you want the same graph structure with new inputs.

Fastlog is different: predicates see lightweight `RecordContext` events during the pass and only selected records are copied to RAM and/or written to disk. Use it for repeated rollouts, targeted probes, memory-constrained activation collection, and training-time auxiliary losses where a full `ModelLog` would be too heavy.


In [ ]:
from __future__ import annotations

import shutil
import sys
from pathlib import Path

import torch
from torch import nn

repo_root = Path.cwd().resolve()
if not (repo_root / "torchlens").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import torchlens as tl  # noqa: E402


torch.manual_seed(7)


class TinyMLP(nn.Module):
    """Small MLP used by most examples."""

    def __init__(self) -> None:
        """Initialize three linear layers with ReLU activations."""

        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 3),
            nn.ReLU(),
            nn.Linear(3, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return logits for one batch."""

        return self.net(x)


model = TinyMLP()
x = torch.randn(5, 4)


def keep_op(ctx: tl.fastlog.RecordContext) -> bool:
    """Keep ReLU operation events."""

    return ctx.kind == "op" and ctx.layer_type == "relu"


recording = tl.fastlog.record(model, x, keep_op=keep_op)
print(recording.summary())

## Cross-layer predicates

`RecordContext.recent_ops` is a bounded history of recent operation events. This lets a predicate ask about local graph context without building a full `ModelLog`. Here the rule keeps a ReLU only when a convolution appeared immediately before it.


In [ ]:
class ConvBlock(nn.Module):
    """Small convolutional block for history-aware predicates."""

    def __init__(self) -> None:
        """Initialize a Conv-ReLU-Flatten-Linear model."""

        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(2 * 4 * 4, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return logits for an image batch."""

        return self.net(x)


def relu_after_conv(ctx: tl.fastlog.RecordContext) -> bool:
    """Keep ReLU outputs whose recent history contains a convolution."""

    return (
        ctx.kind == "op"
        and ctx.layer_type == "relu"
        and any(previous.layer_type == "conv2d" for previous in ctx.recent_ops[-2:])
    )


conv_model = ConvBlock()
x_img = torch.randn(2, 1, 4, 4)
conv_recording = tl.fastlog.record(conv_model, x_img, keep_op=relu_after_conv)
print([record.ctx.label for record in conv_recording])

## Module-event predicates

`keep_module` receives module enter and exit events. A common pattern is to keep module exits, because those correspond to the module forward output.


In [ ]:
def keep_linear_outputs(ctx: tl.fastlog.RecordContext) -> bool:
    """Keep the output event for every Linear module forward."""

    return ctx.kind == "module_exit" and ctx.module_type == "Linear"


module_recording = tl.fastlog.record(model, x, keep_module=keep_linear_outputs)
print([(record.ctx.module_address, record.ctx.kind) for record in module_recording])

## `CaptureSpec` power-user form

Returning `True` uses the default capture policy. Returning a `CaptureSpec` lets a predicate request details such as dtype conversion, device placement, metadata-only records, or gradient retention.


In [ ]:
def keep_relu_bfloat16(ctx: tl.fastlog.RecordContext) -> bool | tl.fastlog.CaptureSpec:
    """Downcast retained ReLU payloads to bfloat16."""

    if ctx.kind == "op" and ctx.layer_type == "relu":
        return tl.fastlog.CaptureSpec(dtype=torch.bfloat16)
    return False


bf16_recording = tl.fastlog.record(model, x, keep_op=keep_relu_bfloat16)
print([record.ram_payload.dtype for record in bf16_recording])

## Many-rollout pattern

Use `Recorder` when the same predicate should run over many explicit forwards. Logging is active only inside each `rec.log(...)` call, so ordinary torch work between calls is not captured.


In [ ]:
loader = [torch.randn(2, 4), torch.randn(3, 4)]

with tl.fastlog.Recorder(model, keep_op=keep_op) as rec:
    for batch_id, batch in enumerate(loader):
        rec.log(batch, sample_id=batch_id)

many_recording = rec.recording
print(many_recording.summary())
print([record.ctx.sample_id for record in many_recording])

## Disk save

`StreamingOptions(bundle_path=..., retain_in_memory=False)` writes a synchronous fastlog bundle and keeps tensor payloads off RAM. Load finalized bundles with `tl.fastlog.load(...)`.


In [ ]:
bundle_path = Path("./acts.tlfast")
if bundle_path.exists():
    shutil.rmtree(bundle_path)

streaming = tl.StreamingOptions(bundle_path="./acts.tlfast", retain_in_memory=False)
disk_recording = tl.fastlog.record(model, x, keep_op=keep_op, streaming=streaming)
loaded = tl.fastlog.load("./acts.tlfast")
print(disk_recording.summary())
print(loaded.summary())

## Preview pass

When you want to inspect a predicate on a full graph first, build a `ModelLog` and call `preview_fastlog(...)`. The preview colors the graph using synthesized fastlog contexts; it is diagnostic and does not save fastlog activations.


In [ ]:
model_log = tl.log_forward_pass(model, x, layers_to_save="all", vis_save_only=True)
dot_source = model_log.preview_fastlog(
    predicate=keep_op,
    vis_save_only=True,
    vis_outpath="/tmp/fastlog_preview",
    vis_fileformat="dot",
)
print(dot_source.splitlines()[0])

## Dry run and repredicate iteration

`dry_run()` executes the model and records event contexts plus keep/skip decisions without retaining tensor payloads. `repredicate(...)` reuses those contexts to test another predicate quickly.


In [ ]:
def other_predicate(ctx: tl.fastlog.RecordContext) -> bool:
    """Keep Linear operation events."""

    return ctx.kind == "op" and ctx.layer_type == "linear"


trace = tl.fastlog.dry_run(model, x, keep_op=keep_op)
print(trace.print_tree())
print(trace.repredicate(other_predicate).summary())

## Bundle recovery

`recover(...)` scans the append-only index and tolerates a partial bundle. Recovered recordings set `recovered=True` and expose any recovery warnings.


In [ ]:
partial_path = Path("./partial_bundle.tlfast")
if partial_path.exists():
    shutil.rmtree(partial_path)
shutil.copytree(bundle_path, partial_path)
(partial_path / "manifest.json").unlink()

recovered = tl.fastlog.recover("./partial_bundle.tlfast")
print(recovered.recovered)
print(recovered.recovery_warnings)
print(recovered.summary())

## Training compatibility

The v1 training highlight is RAM capture with `CaptureSpec(keep_grad=True)`. The retained payload remains attached to autograd, so it can participate in an auxiliary loss.


In [ ]:
train_model = TinyMLP()
train_x = torch.randn(6, 4)


def keep_relu_with_grad(ctx: tl.fastlog.RecordContext) -> bool | tl.fastlog.CaptureSpec:
    """Keep ReLU payloads attached to autograd."""

    if ctx.kind == "op" and ctx.layer_type == "relu":
        return tl.fastlog.CaptureSpec(keep_grad=True)
    return False


output, train_recording = tl.fastlog.record(
    train_model,
    train_x,
    keep_op=keep_relu_with_grad,
    return_output=True,
)
auxiliary_loss = train_recording[0].ram_payload.square().mean()
loss = output.mean() + auxiliary_loss
loss.backward()
assert any(
    parameter.grad is not None and parameter.grad.abs().sum().item() > 0
    for parameter in train_model.parameters()
)
print("gradient reached model parameters")

## Not supported in v1

- DDP forward semantics: fastlog unwraps and captures rank-local `.module` behavior only.
- FSDP: rejected with the existing fastlog error.
- Async disk drain: disk writes are synchronous in v1.
- `Recording.to_model_log()`: sparse captures cannot reconstruct a faithful `ModelLog`.
- `keep_grad` with disk-only sessions: use RAM+disk mirror instead.
